# 地质图 SAM 2.1 分割与 GeoTIFF 矢量化｜Colab 测试版

本 Notebook 使用 [Meta SAM 2](https://github.com/facebookresearch/sam2) 对地质图进行**自动候选区域分割**或**点击/框选分割**，输出叠加预览、Mask、GeoJSON。上传 GeoTIFF 且包含 CRS 时，还会输出保留坐标参考的分区 GeoTIFF 和 WGS84 GeoJSON。

**快速开始**：在 Colab 菜单选择 **运行时 → 更改运行时类型 → T4 GPU**（若账户可用），随后顺序运行全部单元。默认使用程序生成的演示图，不用上传资料。运行到设置单元，将 `SOURCE='upload'` 则可上传 JPG/PNG/GeoTIFF。

**用途边界**：这是算法验证原型。SAM 生成的 Mask 不带地层代号、断层属性，也不直接生成 MapGIS WT/WL/WP。自动候选区域可能遗漏、重叠或误分割，需要复核。地质资料如果具有保密或使用限制，请仅使用经许可的公开测试数据，或在符合要求的本地环境运行，不要上传 Colab。

## 第一步：准备运行环境与SAM 2.1 Tiny权重

从Meta官方仓库获取代码和轻量模型。模型权重只保存在当前临时运行环境，不提交到GitHub。首次运行需下载，Colab断开后可能需要重新执行。

In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request, time

BASE = Path('/content') if Path('/content').exists() else Path.cwd() / '.sam2_runtime'
BASE.mkdir(parents=True, exist_ok=True)
SRC = BASE / 'sam2'
CKPT = BASE / 'sam2.1_hiera_tiny.pt'
CKPT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt'
CFG = 'configs/sam2.1/sam2.1_hiera_t.yaml'

print('Python:', sys.version.split()[0])
if not SRC.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/sam2.git', str(SRC)], check=True)

env = dict(os.environ, SAM2_BUILD_CUDA='0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SRC)], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio>=1.3', 'matplotlib>=3.8', 'Pillow>=9.4', 'opencv-python-headless>=4.7'], check=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # editable安装后，无需重启内核即可import SAM2代码

if not CKPT.exists() or CKPT.stat().st_size < 1_000_000:
    print('下载 SAM 2.1 Tiny 模型权重……')
    urllib.request.urlretrieve(CKPT_URL, CKPT)
assert CKPT.stat().st_size > 1_000_000, '模型权重下载不完整'
print('SAM 2.1 Tiny 权重就绪：', CKPT.name, f'({CKPT.stat().st_size/1024**2:.1f} MB)')

## 第二步：选择演示图或上传真实图像

默认 `SOURCE='demo'` 自动创建一幅**人工合成的测试地质图**，仅用于检验运行流程。要测试真实图片，将 `SOURCE` 修改为 `'upload'`，运行单元后在弹出的窗口上传文件。

对多波段GeoTIFF，默认使用第1、2、3波段生成RGB预览。请根据具体遥感/地质图波段重新指定。大图会按最长边缩小到 `MAX_EDGE`，矢量输出会与该预览网格对应，适合技术原型验证。

In [ ]:
from pathlib import Path
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import rasterio
from rasterio.enums import Resampling
from affine import Affine

SOURCE = 'demo'       # 'demo' / 'upload'
MAX_EDGE = 1280      # 单次推理最大边长，建议 1024~1536
RGB_BANDS = (1, 2, 3)  # 多波段GeoTIFF构成RGB预览的波段编号（1起始）


def create_demo(path):
    im = Image.new('RGB', (900, 650), (244, 232, 183))
    d = ImageDraw.Draw(im)
    d.polygon([(0,0),(450,0),(375,125),(440,250),(320,330),(170,255),(0,270)], fill=(189,216,184))
    d.polygon([(450,0),(900,0),(900,330),(730,300),(660,390),(440,250),(375,125)], fill=(233,194,188))
    d.polygon([(0,270),(170,255),(320,330),(440,250),(660,390),(630,650),(0,650)], fill=(239,221,166))
    d.polygon([(730,300),(900,330),(900,650),(630,650),(660,390)], fill=(181,202,232))
    d.line([(115,5),(250,190),(400,400),(575,650)], fill=(95,55,55), width=6)
    d.text((170,110), 'J3', fill='black'); d.text((640,100),'K1',fill='black')
    d.text((220,500),'P2',fill='black'); d.text((780,500),'N',fill='black')
    im.save(path)
    return path


def _stretch(band):
    x = band.astype('float32')
    ok = np.isfinite(x)
    if not np.any(ok):
        return np.zeros_like(x, dtype='uint8')
    lo, hi = np.percentile(x[ok], (2, 98))
    if hi <= lo:
        return np.zeros_like(x, dtype='uint8')
    return (np.clip((x-lo)/(hi-lo), 0, 1)*255).astype('uint8')


def load_input(path, max_edge=1280, rgb_bands=(1,2,3)):
    path = Path(path)
    if path.suffix.lower() in ('.tif', '.tiff'):
        with rasterio.open(path) as ds:
            scale = min(1, max_edge/max(ds.width, ds.height))
            w = max(1, round(ds.width*scale)); h = max(1, round(ds.height*scale))
            bands = tuple(b for b in rgb_bands if 1 <= b <= ds.count)
            if not bands:
                bands = (1,)
            stack = ds.read(indexes=list(bands), out_shape=(len(bands),h,w), resampling=Resampling.nearest)
            # 单波段带调色板的分类图，优先使用原有图像的颜色。
            cmap = None
            if ds.count == 1:
                try:
                    cmap = ds.colormap(1)
                except ValueError:
                    pass
            if cmap:
                raw = stack[0]
                arr = np.zeros((h,w,3), dtype=np.uint8)
                for k,rgba in cmap.items():
                    arr[raw == k] = rgba[:3]
            elif len(bands) == 1:
                gray = stack[0] if stack.dtype == np.uint8 else _stretch(stack[0])
                arr = np.repeat(gray[...,None], 3, axis=2)
            elif len(bands) == 2:
                channels = [stack[0], stack[1], stack[0]]
                arr = np.stack([b if b.dtype == np.uint8 else _stretch(b) for b in channels], axis=-1)
            else:
                arr = np.stack([b if b.dtype == np.uint8 else _stretch(b) for b in stack[:3]], axis=-1)
            preview_transform = ds.transform * Affine.scale(ds.width/w, ds.height/h)
            crs = ds.crs
            info = {'input_type':'GeoTIFF', 'original_size':[ds.width,ds.height], 'preview_size':[w,h],
                    'source_crs':str(crs) if crs else None, 'bands':list(bands), 'resampled':bool(scale<1)}
            return np.asarray(arr, dtype=np.uint8), preview_transform if crs else None, crs, info

    im = Image.open(path).convert('RGB')
    original = im.size
    im.thumbnail((max_edge,max_edge), Image.Resampling.LANCZOS)
    return np.asarray(im, dtype=np.uint8), None, None, {'input_type':'image', 'original_size':list(original), 'preview_size':list(im.size), 'source_crs':None, 'resampled':im.size!=original}

if SOURCE == 'demo':
    input_path = create_demo(BASE/'demo_geological_map.png')
elif SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('请一次只上传一个PNG/JPG/GeoTIFF文件')
    name, data = next(iter(uploaded.items()))
    input_path = BASE / Path(name).name
    input_path.write_bytes(data)
else:
    raise ValueError('SOURCE必须为demo或upload')

image_rgb, geo_transform, geo_crs, input_info = load_input(input_path, MAX_EDGE, RGB_BANDS)
H, W = image_rgb.shape[:2]
print('输入：', input_path.name, '| 预览尺寸：', W, 'x', H, '| CRS：', str(geo_crs))
plt.figure(figsize=(12,7))
plt.imshow(image_rgb)
plt.title('待分割图像（横纵坐标为当前预览像素）')
plt.xlabel('x / pixel'); plt.ylabel('y / pixel'); plt.grid(alpha=0.2)
plt.show()

## 第三步：运行SAM分割

可选择 `MODE='auto'`（自动寻找候选区域）、`MODE='point'`（点击一个地质体内部）或 `MODE='box'`（点击矩形框左上和右下）。`point` 和 `box` 在Google Colab中会弹出可点击预览图；若不在Colab环境，可改用代码给定的手动坐标。

**建议先运行默认的自动分割**，然后用点提示检查你关心的某个地质体。自动Mask的数量是候选分割结果数量，不等于地质单元数量。

In [ ]:
import torch
from contextlib import nullcontext
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

MODE = 'auto'  # 'auto', 'point', 'box'
AUTO_POINTS_PER_SIDE = 12  # 点数平方；越大越慢，建议12/16
MAX_AUTO_MASKS = 30
MIN_AREA_PIXELS = 120
USE_CLICK_UI = True
MANUAL_POINT = (W//2, H//2) # USE_CLICK_UI=False时使用
MANUAL_BOX = (W//4, H//4, 3*W//4, 3*H//4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('推理设备：', device, ' | 模型：SAM 2.1 Tiny')
model = build_sam2(CFG, str(CKPT), device=device, apply_postprocessing=False)


def click_on_image(prompt):
    # 仅Colab支持交互点击；返回当前预览图对应的像素坐标。
    import base64, json
    from google.colab import output
    buf = io.BytesIO()
    Image.fromarray(image_rgb).save(buf, format='PNG')
    data = base64.b64encode(buf.getvalue()).decode('ascii')
    js = f"""
      new Promise((resolve) => {{
        const wrap = document.createElement('div');
        wrap.style.cssText = 'background:white;padding:12px;border:1px solid #ddd;max-width:900px';
        const hint = document.createElement('p');
        hint.textContent = {json.dumps(prompt)};
        const im = document.createElement('img');
        im.src = 'data:image/png;base64,{data}';
        im.style.cssText = 'max-width:100%;height:auto;cursor:crosshair;display:block';
        wrap.appendChild(hint); wrap.appendChild(im); document.body.appendChild(wrap);
        im.onclick = (ev) => {{
          const r = im.getBoundingClientRect();
          const x = Math.max(0, Math.min({W-1}, Math.round((ev.clientX-r.left)*{W}/r.width)));
          const y = Math.max(0, Math.min({H-1}, Math.round((ev.clientY-r.top)*{H}/r.height)));
          wrap.remove(); resolve([x,y]);
        }};
      }})
    """
    return tuple(map(int, output.eval_js(js)))

mask_records = []
with torch.inference_mode():
    if MODE == 'auto':
        generator = SAM2AutomaticMaskGenerator(
            model=model, points_per_side=AUTO_POINTS_PER_SIDE, points_per_batch=16,
            pred_iou_thresh=0.75, stability_score_thresh=0.85,
            min_mask_region_area=0, crop_n_layers=0,
        )
        candidates = generator.generate(image_rgb)
        candidates = sorted(candidates, key=lambda r: float(r['predicted_iou']), reverse=True)
        occupied = np.zeros((H,W), dtype=bool)
        for r in candidates:
            m = np.asarray(r['segmentation'], dtype=bool) & ~occupied
            area = int(m.sum())
            if area < MIN_AREA_PIXELS or area > int(H*W*0.85):
                continue
            occupied |= m
            mask_records.append({'mask':m, 'score':float(r['predicted_iou']), 'pixel_area':area})
            if len(mask_records) >= MAX_AUTO_MASKS:
                break
    elif MODE in ('point','box'):
        predictor = SAM2ImagePredictor(model)
        predictor.set_image(image_rgb)
        if MODE == 'point':
            p = click_on_image('单击一个地质体内部以提取轮廓') if USE_CLICK_UI else MANUAL_POINT
            kwargs = {'point_coords':np.asarray([p], dtype=np.float32), 'point_labels':np.asarray([1], dtype=np.int32)}
            print('点提示：',p)
        else:
            if USE_CLICK_UI:
                a = click_on_image('单击目标区域的左上角')
                b = click_on_image('单击目标区域的右下角')
            else:
                a,b = MANUAL_BOX[:2], MANUAL_BOX[2:]
            x0,x1=sorted((a[0],b[0])); y0,y1=sorted((a[1],b[1]))
            kwargs={'box':np.asarray([x0,y0,x1,y1],dtype=np.float32)}
            print('框提示：', kwargs['box'])
        masks, scores, _ = predictor.predict(multimask_output=True, **kwargs)
        best = int(np.argmax(scores)); m = np.asarray(masks[best],dtype=bool)
        if int(m.sum()) >= MIN_AREA_PIXELS:
            mask_records = [{'mask':m, 'score':float(scores[best]), 'pixel_area':int(m.sum())}]
    else:
        raise ValueError('MODE必须为auto、point或box')

print(f'候选区域数量：{len(mask_records)}')
if not mask_records:
    print('未获得有效分割区域：可降低MIN_AREA_PIXELS，或改用point/box提示。')

rng = np.random.default_rng(42)
preview = image_rgb.astype('float32').copy()
for r in mask_records:
    color = rng.integers(55,255,size=3)
    preview[r['mask']] = preview[r['mask']]*0.45 + color*0.55
preview = preview.clip(0,255).astype('uint8')

plt.figure(figsize=(12,7)); plt.imshow(preview)
plt.title(f'SAM分割叠加预览 | 模式：{MODE} | 区域：{len(mask_records)}')
plt.axis('off'); plt.show()

## 第四步：导出Mask、GeoJSON和结果压缩包

- **输入是带CRS的GeoTIFF**：输出与预览网格匹配的 `sam_regions_georeferenced.tif`、EPSG:4326的 `sam_polygons_wgs84.geojson`。
- **输入是PNG/JPG或无CRS的TIFF**：输出 `sam_regions_pixels.png` 和 `sam_polygons_pixel_coordinates.geojson`。后者的坐标是像素坐标，不能直接作为真实地理位置使用。
- 保存 `sam_overlay.png`、`summary.json` 和 `sam_results.zip`。

**注意**：这一步输出的是候选面要素。它们不含地质体名称、地层代号和拓扑检查结果，尚未生成MapGIS文件。

In [ ]:
import json, zipfile, shutil
from rasterio.features import shapes
from rasterio.warp import transform_geom
from rasterio.transform import Affine as RasterAffine

OUT = BASE/'sam2_geo_results'
if OUT.exists():
    shutil.rmtree(OUT)  # 仅清除该Notebook自己的上次输出，防止旧数据混入压缩包
OUT.mkdir(parents=True, exist_ok=True)
Image.fromarray(preview).save(OUT/'sam_overlay.png')

labels = np.zeros((H,W),dtype=np.uint16)
features = []

for idx,rec in enumerate(mask_records, start=1):
    # 自动模式下已经进行掩膜去重；交互模式只含一个目标。
    m = rec['mask'] & (labels == 0)
    labels[m] = idx
    for poly, value in shapes(m.astype('uint8'), mask=m, transform=geo_transform or RasterAffine.identity()):
        if value != 1: continue
        if geo_crs:
            poly = transform_geom(geo_crs, 'EPSG:4326', poly, precision=8)
        features.append({'type':'Feature', 'geometry':poly,
                         'properties':{'mask_id':idx, 'score':round(rec['score'],4),
                                       'mask_pixel_area':rec['pixel_area']}})

if geo_crs:
    raster_path = OUT/'sam_regions_georeferenced.tif'
    with rasterio.open(raster_path, 'w', driver='GTiff', height=H, width=W, count=1,
                       dtype='uint16', crs=geo_crs, transform=geo_transform,
                       nodata=0, compress='lzw') as dst:
        dst.write(labels,1)
    json_path = OUT/'sam_polygons_wgs84.geojson'
else:
    Image.fromarray(labels).save(OUT/'sam_regions_pixels.png')
    json_path = OUT/'sam_polygons_pixel_coordinates.geojson'

fc = {'type':'FeatureCollection', 'features':features}
# GeoJSON坐标为EPSG:4326；无坐标系时额外写明这只是像素坐标。
if not geo_crs:
    fc['metadata'] = {'coordinate_system':'pixel', 'warning':'not georeferenced; pixel coordinates only'}
json_path.write_text(json.dumps(fc, ensure_ascii=False), encoding='utf-8')
summary = {'model':'Meta SAM 2.1 Hiera Tiny', 'mode':MODE, 'mask_count':len(mask_records),
           'polygon_count':len(features), 'geo_reference':bool(geo_crs), **input_info}
(OUT/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = BASE/'sam_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.iterdir()): zf.write(p, arcname=p.name)
print('已保存：', zip_path)
print('分割区域：',len(mask_records),'矢量面：',len(features))
print('导出文件：', [p.name for p in OUT.iterdir()])

# 在Colab中触发下载。若本地运行，可从BASE文件夹读取结果。
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('当前不是Colab，结果文件保留在本地目录。')

## 第五步：如何评价分割是否可用于地质图转换

1. 是否把相同地质体分割成合理的连续区域？
2. 有无边缘错位、地质符号导致的孔洞、过度分割或漏分？
3. 输出图斑的像素/地理坐标是否正确？
4. 哪些地质体需要补充点提示、图例语义或人工修改？

**下一阶段**：与现有PaddleOCR文字框建立空间关联，为图斑赋予地层代号及其他属性；对地质界线和断层单独识别；完成拓扑清理；再研发MapGIS的WT/WL/WP/MPJ写出和回读验证。